In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
import io
from scipy.stats import multivariate_normal
DV01 = 17
meeting_data = {
        'Meeting_Date': pd.to_datetime([
        "16/06/2026", "04/08/2026", "15/09/2026", "03/11/2026", "08/12/2026",
        "09/02/2027", "23/03/2027", "04/05/2027", "22/06/2027", "10/08/2027",
        "21/09/2027", "02/11/2027", "14/12/2027", '08/02/2028'
    ], format="%d/%m/%Y"),
        'Meeting_ID': [
        'jun26_meeting', 'aug26_meeting', 'sep26_meeting', 'nov26_meeting', 'dec26_meeting',
        'feb27_meeting', 'mar27_meeting', 'may27_meeting', 'jun27_meeting', 'aug27_meeting',
        'sep27_meeting', 'nov27_meeting', 'dec_27_meeting', 'feb28_meeting']
}
df_meetings = pd.DataFrame(meeting_data)
contract_data = {

    'Expiry_Date': pd.to_datetime(["11/06/2026", "10/09/2026", "10/12/2026", "11/03/2027", "10/06/2027", "09/09/2027", "09/12/2027" ],format="%d/%m/%Y"),
    'Contracts': ['IR Jun26', 'IR Sep26', 'IR Dec26', 'IR Mar27', 'IR Jun27', 'IR Sep27', 'IR Dec27'],
}
df_contracts = pd.DataFrame(contract_data)

meeting_ids = ['jun26_meeting', 'aug26_meeting', 'sep26_meeting', 'nov26_meeting', 'dec26_meeting',
        'feb27_meeting', 'mar27_meeting', 'may27_meeting', 'jun27_meeting', 'aug27_meeting',
        'sep27_meeting', 'nov27_meeting', 'dec_27_meeting', 'feb28_meeting']


# Prompt user to upload Position

print("📂 Please upload your positions file (CSV or Excel)")
uploaded = files.upload()
filename = next(iter(uploaded))
if filename.lower().endswith('.csv'):
    df_positions = pd.read_csv(io.BytesIO(uploaded[filename]))
elif filename.lower().endswith(('.xlsx', '.xls')):
    df_positions = pd.read_excel(io.BytesIO(uploaded[filename]))
else:
    raise ValueError("Please upload a CSV or Excel file.")
df_positions = df_positions.dropna(how = 'all', axis=1)
df_positions = df_positions.dropna(subset = [df_positions.columns[0]])
positions_array = df_positions.iloc[:,1].to_numpy()

# Creating Impact Matrix
df_matrix = pd.merge(df_meetings, df_contracts,  how='cross')
day_diff = (df_matrix["Meeting_Date"] - df_matrix["Expiry_Date"]).dt.days
conditions = [df_matrix['Meeting_Date']<df_matrix['Expiry_Date'], (df_matrix['Meeting_Date']>=df_matrix['Expiry_Date']) & (day_diff <= 90) ]
choice = [1, round((90-day_diff)/90, 3)]
df_matrix["Impact"] = np.select(conditions,choice, default = 0 )
impact_matrix = df_matrix.pivot(index = 'Contracts', columns = 'Meeting_ID', values = 'Impact')

impact_matrix = impact_matrix.reindex(index = df_contracts['Contracts'], columns = df_meetings['Meeting_ID'])
impact_matrix = impact_matrix.reset_index().rename_axis(None, axis=1)
impact_filter = impact_matrix.loc[impact_matrix['Contracts'].isin(df_positions['Contracts'])]
impact_array = impact_filter.iloc[:, 1:].to_numpy()
impact_array = positions_array.T @ impact_array

#Prompt user to upload live premiums and scenario premiums
print("📂 Please upload your Scenarios with live premium file (CSV or Excel)")
uploaded_scenarios = files.upload()
filename_scenarios = next(iter(uploaded_scenarios))
if filename_scenarios.lower().endswith('.csv'):
  df_premiums = pd.read_csv(io.BytesIO(uploaded_scenarios[filename_scenarios]))
elif filename_scenarios.lower().endswith(('.xlsx', '.xls')):
  df_premiums = pd.read_excel(io.BytesIO(uploaded_scenarios[filename_scenarios]))
else:
  raise ValueError("Please upload a CSV or Excel file.")
df_premiums.iloc[:,1:] = df_premiums.iloc[:,1:].apply(pd.to_numeric, errors='coerce')
df_change = df_premiums.iloc[:,2:].sub(df_premiums.iloc[:,1], axis=0)


#Prompt user if he wants to use statistical probability
probability_choice = input("Do you want to use statistical probability? (Y/N)")
scenario_probability = {}
if probability_choice.upper() == "Y":
  print("📂 Please upload your historical meeting premiums file (CSV or Excel)")
  uploaded_historical_premiums = files.upload()
  filename_historical_premiums = next(iter(uploaded_historical_premiums))
  if filename_historical_premiums.lower().endswith('.csv'):
    df_historical = pd.read_csv(io.BytesIO(uploaded_historical_premiums[filename_historical_premiums]))
  elif filename_historical_premiums.lower().endswith(('.xlsx', '.xls')):
    df_historical = pd.read_excel(io.BytesIO(uploaded_historical_premiums[filename_historical_premiums]))
  else:
    raise ValueError("Please upload a CSV or Excel file.")
  df_trimmed = df_historical.iloc[:,1:]
  historical_matrix = df_trimmed.to_numpy()
  mu = np.mean(historical_matrix, axis=0)
  sigma = np.cov(historical_matrix, rowvar=False)
  sigma += np.eye(14)*1e-6
  mvn = multivariate_normal(mean = mu, cov = sigma, allow_singular = True)
  for col in df_premiums.columns[2:]:
    view_vector = df_premiums[col].to_numpy().astype(float)
    scenario_probability[col] = mvn.pdf(view_vector)
  total_probabilty = sum(scenario_probability.values())
  for key, value in scenario_probability.items():
    scenario_probability[key] = round(float(value/total_probabilty), 4)
else:
  for col in df_premiums.columns[2:]:
    scenario_probability[col] = 1/len(df_premiums.columns[2:])


#Function for PNL Fluctuation
def pnl_cal(change_vector):
    v = change_vector.to_numpy().flatten().astype(float)
    exposure = impact_array.flatten()
    pnl_scalar = np.dot(exposure, v)
    total_change = float(-pnl_scalar * DV01)
    return total_change

expected_pnl = 0
print("\n" + "="*65)
print(f"{'Scenario Name':<15} | {'Assigned Weight':<15} | {'Calculated Scenario PnL':<20}")
print("-"*65)

for key, weight in scenario_probability.items():
    scenario_pnl = pnl_cal(df_change[key])
    expected_pnl += scenario_pnl * weight
    print(f"{key:<15} | {weight*100:<13.2f}% | ${scenario_pnl:,.2f}")

print("="*65)
print(f"👉 YOUR TOTAL WEIGHTED EXPECTED PORTFOLIO PNL IS: ${expected_pnl:,.2f} USD")
print("="*65)

📂 Please upload your positions file (CSV or Excel)


Saving Data.csv to Data (5).csv
📂 Please upload your Scenarios with live premium file (CSV or Excel)


Saving Scenarios.csv to Scenarios (2).csv
Do you want to use statistical probability? (Y/N)Y
📂 Please upload your historical meeting premiums file (CSV or Excel)


Saving historical_data.xlsx to historical_data (1).xlsx

Scenario Name   | Assigned Weight | Calculated Scenario PnL
-----------------------------------------------------------------
Scenario A      | 3.47         % | $2,060.84
Scenario B      | 38.54        % | $-1,392.55
Scenario C      | 57.99        % | $-2,768.81
👉 YOUR TOTAL WEIGHTED EXPECTED PORTFOLIO PNL IS: $-2,070.81 USD


In [ ]:
 #Filter impact matrix as per positions
positions_array = df_positions.iloc[:,1].to_numpy() #Array Creation for positions
impact_array = impact_filter.iloc[:, 1:].to_numpy() #Array Creation for impact
impact_array = positions_array.T @ impact_array     #Matrix multiplication of positions and impact
active_impact_array = impact_array[impact_array != 0] #Selecting only active impact meetings
total_change_lst = []
for i in len(df_change):
  change_array = df_change.iloc[:, i].to_numpy()
  total_change = 0 - (impact_array @ change_array.reshape(-1,1))*DV01
  total_change_lst.append(total_change)

print ("Your total PNL fluctuation is", total_change[0], "$")
